# Fine-tune PhoBERT for Vietnamese Relation Extraction

**Model:** `vinai/phobert-base-v2`  
**Input:** `/kaggle/input/<dataset>/triples.json`  
**Output:** `/kaggle/working/phobert_re/best_model/`

**Pipeline:**
1. Load & filter triples by confidence
2. Map VI labels → canonical EN labels
3. Insert Entity Markers into source_text
4. Generate Negative samples (No_Relation)
5. Train/Val/Test split → Fine-tune → Evaluate → Save

## 1. Install dependencies

In [55]:
# Kaggle đã có sẵn torch, transformers — chỉ cần upgrade nếu cần
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers>=4.41", "accelerate", "scikit-learn"],
               check=True)

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'transformers>=4.41', 'accelerate', 'scikit-learn'], returncode=0)

## 2. Imports & Config

In [56]:
import json, random, re, logging
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)

# ── Kaggle paths ──────────────────────────────────────────────────
# Auto-detect triples.json from /kaggle/input/* (no need to change slug)
import glob as _glob
_candidates = _glob.glob("/kaggle/input/**/triples.json", recursive=True)
if _candidates:
    TRIPLES_JSON = Path(_candidates[0])
    print("Found dataset:", TRIPLES_JSON)
else:
    TRIPLES_JSON = Path("triples.json")   # fallback local / workspace
    print("WARNING: /kaggle/input not found, using local:", TRIPLES_JSON)
OUTPUT_DIR = Path("/kaggle/working/phobert_re")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Hyperparams ───────────────────────────────────────────────────
MODEL_NAME     = "vinai/phobert-base-v2"
MAX_LEN        = 256
BATCH_SIZE     = 16
EPOCHS         = 5
LR             = 2e-5
WEIGHT_DECAY   = 0.01
SEED           = 42
MIN_CONFIDENCE = 0.75
NEG_RATIO      = 2

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print("CUDA:", torch.cuda.is_available())

Found dataset: /kaggle/input/datasets/nvt301004/triples/triples.json
CUDA: True


## 3. Relation Map (VI → EN)  
Nhãn `None` sẽ bị loại khỏi training.

In [57]:
RELATION_MAP = {
    "LÃNH_ĐẠO":         "leader_of",
    "HỢP_TÁC":          "partnered_with",
    "THUỘC_NGÀNH":       "operates_in",
    "CÓ_GIÁ_TRỊ":      "has_value",      
    "TĂNG_TRƯỞNG":      "has_growth",     
    "ĐẶT_TẠI":          None,
    "SẢN_XUẤT":         None,
    "TỔ_CHỨC_SỰ_KIỆN":  None,
    "XẢY_RA_TẠI":       None,
    "THAM_GIA":         None,
    "LIÊN_QUAN":         None,
    "ĐẦU_TƯ":           None,
}


ENTITY_TYPES = {
    "PERSON":"PER","ORGANIZATION":"ORG","LOCATION":"LOC",
    "PRODUCT":"PRD","EVENT":"EVT","DATE":"DAT",
    "MONEY":"MON","PERCENT":"PCT","INDUSTRY":"IND","UNKNOWN":"UNK",
}

In [58]:
# DEBUG: xem từng relation bị loại ở bước nào
import json
from pathlib import Path

with open(TRIPLES_JSON, encoding="utf-8") as f:
    raw = json.load(f)
triples = raw.get("triples", raw) if isinstance(raw, dict) else raw
df_all = pd.DataFrame(triples)
df_all.columns = [c.lstrip("\ufeff") for c in df_all.columns]

for rel, label in RELATION_MAP.items():
    if label is None:
        continue
    sub = df_all[df_all["relation"] == rel]
    after_conf = sub[sub["confidence"].astype(float) >= MIN_CONFIDENCE]
    after_text = after_conf[after_conf.apply(
        lambda r: (
            str(r["subject"]).strip().lower() in str(r["source_text"]).lower() and
            str(r["object"]).strip().lower()  in str(r["source_text"]).lower()
        ), axis=1
    )]
    print(f"{rel:25} → total={len(sub):4d} | pass_conf={len(after_conf):4d} | pass_text={len(after_text):4d}  [{label}]")


LÃNH_ĐẠO                  → total=1835 | pass_conf=1835 | pass_text= 669  [leader_of]
HỢP_TÁC                   → total=1002 | pass_conf=1002 | pass_text= 408  [partnered_with]
THUỘC_NGÀNH               → total= 270 | pass_conf=   0 | pass_text=   0  [operates_in]
CÓ_GIÁ_TRỊ                → total=1528 | pass_conf= 985 | pass_text= 209  [has_value]
TĂNG_TRƯỞNG               → total=1361 | pass_conf=   0 | pass_text=   0  [has_growth]


## 4. Load & Filter Triples

In [59]:
with open(TRIPLES_JSON, encoding="utf-8") as f:
    raw = json.load(f)
triples = raw.get("triples", raw) if isinstance(raw, dict) else raw
df = pd.DataFrame(triples)
df.columns = [c.lstrip("\ufeff") for c in df.columns]
print("Total triples:", len(df))

# Filter
df = df[df["confidence"].astype(float) >= MIN_CONFIDENCE]
df = df[df["source_text"].notna() & (df["source_text"].str.strip() != "")]
df["label"] = df["relation"].map(lambda r: RELATION_MAP.get(r))
df = df[df["label"].notna()].reset_index(drop=True)
df = df[df.apply(
    lambda r: (
        str(r["subject"]).strip().lower() in str(r["source_text"]).lower() and
        str(r["object"]).strip().lower()  in str(r["source_text"]).lower()
    ), axis=1
)].reset_index(drop=True)

print(f"After filtering: {len(df)} rows")
print(df["label"].value_counts().to_string())

Total triples: 6298
After filtering: 1286 rows
label
leader_of         669
partnered_with    408
has_value         209


## 5. Insert Entity Markers

In [60]:
def _tag(ner): return ENTITY_TYPES.get(str(ner).upper(), "UNK")

def insert_markers(text, subj, s_type, obj, o_type):
    st, st_ = f"[SUBJ-{_tag(s_type)}]", f"[/SUBJ-{_tag(s_type)}]"
    ot, ot_ = f"[OBJ-{_tag(o_type)}]",  f"[/OBJ-{_tag(o_type)}]"
    tl = text.lower()
    si = tl.find(str(subj).strip().lower())
    oi = tl.find(str(obj).strip().lower())
    if si == -1 or oi == -1: return None
    entities = sorted(
        [(si, si+len(subj), st, st_, text[si:si+len(subj)]),
         (oi, oi+len(obj),  ot, ot_, text[oi:oi+len(obj)])],
        key=lambda x: -x[0])
    res = text
    for s, e, op, cl, _ in entities:
        res = res[:s] + op + res[s:e] + cl + res[e:]
    return res

# Build positives
positives = []
for _, row in df.iterrows():
    m = insert_markers(str(row["source_text"]),
                       str(row["subject"]), str(row["subject_type"]),
                       str(row["object"]),  str(row["object_type"]))
    if m: positives.append({"text": m, "label": str(row["label"])})
print("Positive samples:", len(positives))

Positive samples: 1286


## 6. Generate Negative Samples

In [61]:
pos_set_by_src = {}
for _, row in df.iterrows():
    sid = str(row.get("source_id", row["source_text"][:40]))
    pos_set_by_src.setdefault(sid, set()).add(
        (str(row["subject"]).lower(), str(row["object"]).lower()))

negatives = []
for src_id, grp in df.groupby(df.get("source_id", df.index).astype(str)):
    text = str(grp.iloc[0]["source_text"])
    pos_set = pos_set_by_src.get(str(src_id), set())
    ents = list({(str(r["subject"]), str(r["subject_type"])): None for _,r in grp.iterrows()}.keys()) \
         + list({(str(r["object"]),  str(r["object_type"])): None for _,r in grp.iterrows()}.keys())
    if len(ents) < 2: continue
    gen, att = 0, 0
    while gen < NEG_RATIO and att < 20:
        att += 1
        a, b = random.sample(ents, 2)
        if (a[0].lower(), b[0].lower()) in pos_set: continue
        m = insert_markers(text, a[0], a[1], b[0], b[1])
        if m is None: continue
        negatives.append({"text": m, "label": "No_Relation"})
        gen += 1
print("Negative samples:", len(negatives))

Negative samples: 1935


## 7. Train/Val/Test Split & Label Encoding

In [62]:
all_samples = positives + negatives
random.shuffle(all_samples)
texts      = [s["text"]  for s in all_samples]
labels_str = [s["label"] for s in all_samples]

label_set  = sorted(set(labels_str))
label2id   = {l: i for i, l in enumerate(label_set)}
id2label   = {i: l for l, i in label2id.items()}
labels_int = [label2id[l] for l in labels_str]

print("Label distribution:")
for lbl, cnt in Counter(labels_str).most_common():
    print(f"  {lbl:<30}: {cnt}")

# Save label map
with open(OUTPUT_DIR / "label_map.json", "w", encoding="utf-8") as f:
    json.dump({"label2id": label2id, "id2label": id2label}, f, ensure_ascii=False, indent=2)

X_temp, X_test, y_temp, y_test = train_test_split(
    texts, labels_int, test_size=0.10, random_state=SEED, stratify=labels_int)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,  test_size=0.11, random_state=SEED, stratify=y_temp)
print(f"Train:{len(X_train)} | Val:{len(X_val)} | Test:{len(X_test)}")

Label distribution:
  No_Relation                   : 1935
  leader_of                     : 669
  partnered_with                : 408
  has_value                     : 209
Train:2579 | Val:319 | Test:323


## 8. Tokenize with PhoBERT

In [63]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

special_tokens = []
for tag in ENTITY_TYPES.values():
    special_tokens += [f"[SUBJ-{tag}]", f"[/SUBJ-{tag}]",
                       f"[OBJ-{tag}]",  f"[/OBJ-{tag}]"]
tokenizer.add_special_tokens({"additional_special_tokens": special_tokens})
print(f"Vocab size after adding {len(special_tokens)} special tokens:", len(tokenizer))

class REDataset(torch.utils.data.Dataset):
    def __init__(self, enc, labs):
        self.enc, self.labs = enc, labs
    def __len__(self): return len(self.labs)
    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.labs[i])
        return item

def tok(lst):
    return tokenizer(lst, padding="max_length", truncation=True,
                     max_length=MAX_LEN, return_tensors="np")

train_ds = REDataset(tok(X_train), y_train)
val_ds   = REDataset(tok(X_val),   y_val)
test_ds  = REDataset(tok(X_test),  y_test)

2026-05-13 12:24:28,690 [INFO] HTTP Request: HEAD https://huggingface.co/vinai/phobert-base-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-13 12:24:28,705 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/vinai/phobert-base-v2/e2375d266bdf39c6e8e9a87af16a5da3190b0cc8/config.json "HTTP/1.1 200 OK"
2026-05-13 12:24:28,767 [INFO] HTTP Request: HEAD https://huggingface.co/vinai/phobert-base-v2/resolve/main/tokenizer_config.json "HTTP/1.1 404 Not Found"
2026-05-13 12:24:28,829 [INFO] HTTP Request: HEAD https://huggingface.co/vinai/phobert-base-v2/resolve/main/tokenizer_config.json "HTTP/1.1 404 Not Found"
2026-05-13 12:24:28,890 [INFO] HTTP Request: GET https://huggingface.co/api/models/vinai/phobert-base-v2/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-05-13 12:24:28,955 [INFO] HTTP Request: GET https://huggingface.co/api/models/vinai/phobert-base-v2/tree/main?recursive=true&expand=false "HTTP/1

Vocab size after adding 40 special tokens: 64041


## 9. Load Model

In [64]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(label_set),
    id2label=id2label, label2id=label2id,
    ignore_mismatched_sizes=True,
)
model.resize_token_embeddings(len(tokenizer))
total_params = sum(p.numel() for p in model.parameters())
print(f"Model params: {total_params:,}")

2026-05-13 12:24:32,305 [INFO] HTTP Request: HEAD https://huggingface.co/vinai/phobert-base-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-13 12:24:32,321 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/vinai/phobert-base-v2/e2375d266bdf39c6e8e9a87af16a5da3190b0cc8/config.json "HTTP/1.1 200 OK"
2026-05-13 12:24:32,387 [INFO] HTTP Request: HEAD https://huggingface.co/vinai/phobert-base-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-13 12:24:32,403 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/vinai/phobert-base-v2/e2375d266bdf39c6e8e9a87af16a5da3190b0cc8/config.json "HTTP/1.1 200 OK"
2026-05-13 12:24:32,470 [INFO] HTTP Request: HEAD https://huggingface.co/vinai/phobert-base-v2/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
2026-05-13 12:24:32,534 [INFO] HTTP Request: GET https://huggingface.co/api/models/vinai/phobert-base-v2 "HTTP/1.1 200 OK"


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

2026-05-13 12:24:32,655 [INFO] HTTP Request: GET https://huggingface.co/api/models/vinai/phobert-base-v2/commits/main "HTTP/1.1 200 OK"
RobertaForSequenceClassification LOAD REPORT from: vinai/phobert-base-v2
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider t

Model params: 135,032,068


## 10. Fine-tune

In [65]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
        "accuracy": accuracy_score(labels, preds),
    }

args = TrainingArguments(
    output_dir              = str(OUTPUT_DIR / "checkpoints"),
    num_train_epochs        = EPOCHS,
    per_device_train_batch_size  = BATCH_SIZE,
    per_device_eval_batch_size   = BATCH_SIZE * 2,
    learning_rate           = LR,
    weight_decay            = WEIGHT_DECAY,
    lr_scheduler_type       = "cosine",
    warmup_ratio            = 0.1,
    eval_strategy           = "epoch",
    save_strategy           = "epoch",
    load_best_model_at_end  = True,
    metric_for_best_model   = "f1_macro",
    greater_is_better       = True,
    logging_steps           = 50,
    report_to               = "none",
    seed                    = SEED,
    fp16                    = torch.cuda.is_available(),
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,F1 Macro,Accuracy
1,2.290800,1.422699,0.412453,0.689655
2,0.922464,0.453512,0.734506,0.931034
3,0.482106,0.291284,0.737037,0.934169
4,0.318303,0.204611,0.951259,0.974922
5,0.247960,0.171159,0.960379,0.981191


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=405, training_loss=0.8015980970712355, metrics={'train_runtime': 375.1603, 'train_samples_per_second': 34.372, 'train_steps_per_second': 1.08, 'total_flos': 1696438992168960.0, 'train_loss': 0.8015980970712355, 'epoch': 5.0})

## 11. Evaluate on Test Set

In [66]:
preds_out   = trainer.predict(test_ds)
preds       = np.argmax(preds_out.predictions, axis=-1)
all_labels = list(range(len(label_set)))
target_names = [id2label[i] for i in all_labels]

print(classification_report(y_test, preds, labels=all_labels, target_names=target_names, zero_division=0))
print("F1 macro :", f1_score(y_test, preds, average="macro", zero_division=0))
print("Accuracy :", accuracy_score(y_test, preds))

cm = confusion_matrix(y_test, preds, labels=all_labels)
print(pd.DataFrame(cm, index=target_names, columns=target_names))

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


                precision    recall  f1-score   support

   No_Relation       0.98      0.97      0.98       194
     has_value       0.82      0.86      0.84        21
     leader_of       0.99      1.00      0.99        67
partnered_with       1.00      1.00      1.00        41

      accuracy                           0.98       323
     macro avg       0.95      0.96      0.95       323
  weighted avg       0.98      0.98      0.98       323

F1 macro : 0.9522691265792844
Accuracy : 0.9752321981424149
                No_Relation  has_value  leader_of  partnered_with
No_Relation             189          4          1               0
has_value                 3         18          0               0
leader_of                 0          0         67               0
partnered_with            0          0          0              41


In [67]:
best_dir = OUTPUT_DIR / "best_model"
trainer.save_model(str(best_dir))
tokenizer.save_pretrained(str(best_dir))

meta = {
    "base_model":    MODEL_NAME,
    "max_len":       MAX_LEN,
    "num_labels":    len(label_set),
    "label2id":      label2id,
    "id2label":      id2label,
    "special_tokens": special_tokens,
    
    "min_confidence": MIN_CONFIDENCE,
}
with open(best_dir / "metadata.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print("[OK] Saved to", best_dir)
print("Files:", sorted(p.name for p in best_dir.iterdir()))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[OK] Saved to /kaggle/working/phobert_re/best_model
Files: ['added_tokens.json', 'bpe.codes', 'config.json', 'metadata.json', 'model.safetensors', 'tokenizer_config.json', 'training_args.bin', 'vocab.txt']


## 12. Save Model

In [68]:
import shutil
from IPython.display import FileLink

zip_name = "phobert_re_best_model"
zip_path = str(OUTPUT_DIR / zip_name)
shutil.make_archive(zip_path, "zip", best_dir)

print(f"Zipped model to: {zip_path}.zip")
display(FileLink(f"phobert_re/{zip_name}.zip"))

Zipped model to: /kaggle/working/phobert_re/phobert_re_best_model.zip


/kaggle/working/phobert_re/phobert_re_best_model.zip